In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('netflix1.csv')
df.head(10)

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,9/24/2021,2021,TV-14,9 Seasons,"British TV Shows, Reality TV"
6,s10,Movie,The Starling,Theodore Melfi,United States,9/24/2021,2021,PG-13,104 min,"Comedies, Dramas"
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,5/1/2021,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu..."
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,9/23/2021,2021,TV-MA,127 min,"Dramas, International Movies"
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,5/1/2021,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals"


In [3]:
print(df.shape)


(8790, 10)


In [4]:
print(df.columns)


Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'],
      dtype='object')


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB
None


In [6]:
print(df.isnull().sum())

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


In [7]:
print("director -> Not Given:", (df["director"].astype(str).str.strip() == "Not Given").sum())
print("country -> Not Given:", (df["country"].astype(str).str.strip() == "Not Given").sum())
print("title -> Unknown:", (df["title"].astype(str).str.strip() == "Unknown").sum())

director -> Not Given: 2588
country -> Not Given: 287
title -> Unknown: 1


In [8]:
print("Full duplicates:", df.duplicated().sum())
print("Duplicates excluding show_id:", df.duplicated(subset=[col for col in df.columns if col != "show_id"]).sum())

Full duplicates: 0
Duplicates excluding show_id: 3


In [9]:
df = df.replace({
    "Not Given": np.nan,
    "Unknown": np.nan
})

In [10]:
print(df.isnull().sum())

show_id            0
type               0
title              1
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


In [11]:
print(df[df["title"].isnull()])

     show_id   type title            director         country date_added  \
1336   s1468  Movie   NaN  Jaume Collet-Serra  United Kingdom   1/1/2021   

      release_year rating duration                      listed_in  
1336          2011  PG-13  113 min  Action & Adventure, Thrillers  


In [12]:
df = df.dropna(subset=["title"])

In [13]:
print(df[["director", "country"]].isnull().sum())

director    2588
country      287
dtype: int64


In [14]:
df["director"] = df["director"].fillna("Unknown Director")
df["country"] = df["country"].fillna("Unknown Country")

In [15]:
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

In [16]:
df = df.drop_duplicates(subset=[col for col in df.columns if col != "show_id"])

In [17]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

In [18]:
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

In [19]:
df["type"] = df["type"].str.title()

In [20]:
df["rating"] = df["rating"].str.upper()

In [21]:
df["listed_in"] = df["listed_in"].str.strip()

In [44]:
df['duration_movies'] = df['duration'].where(df['duration'].str.contains('min', na=False))
df['duration_shows']  = df['duration'].where(df['duration'].str.contains('Season', na=False))

In [50]:
df["duration_movies"] = df["duration_movies"].fillna("Not Applicable")
df["duration_shows"] = df["duration_shows"].fillna("Not Applicable")

In [52]:
df["duration_num"] = df["duration"].str.extract(r"(\d+)")[0]
df["duration_num"] = pd.to_numeric(df["duration_num"], errors="coerce")

df["duration_unit"] = df["duration"].str.extract(r"([A-Za-z]+)")[0]
df["duration_unit"] = df["duration_unit"].str.lower()


In [54]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates excluding show_id:")
print(df.duplicated(subset=[col for col in df.columns if col != "show_id"]).sum())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

Shape: (8785, 14)

Missing values:
show_id            0
type               0
title              0
director           0
country            0
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
duration_movies    0
duration_shows     0
duration_num       0
duration_unit      0
dtype: int64

Duplicates excluding show_id:
0

Data types:
show_id                    object
type                       object
title                      object
director                   object
country                    object
date_added         datetime64[ns]
release_year                int64
rating                     object
duration                   object
listed_in                  object
duration_movies            object
duration_shows             object
duration_num                int64
duration_unit              object
dtype: object

First 5 rows:
  show_id     type                             title         director  \
0      s1    Movie              Di